# 03 — Evaluation: Business-Cost Threshold Tuning

This is the most important notebook — it shows **threshold tuning** using actual maintenance economics.

Interview story:
> 'Standard ML defaults to 0.5. For failure prediction, a missed failure costs $50K in unplanned downtime,
> while a false alarm costs $2K in unnecessary maintenance. I built a cost model and found the threshold
> that minimizes total operational cost — it shifted from 0.5 to ~0.25.'

Contents:
1. Confusion matrix at default (0.5) and cost-optimal threshold
2. ROC curve
3. Precision-recall curve
4. Business cost curve — total cost vs. threshold
5. Summary table

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, roc_curve, precision_recall_curve,
    roc_auc_score, average_precision_score,
)
import sys; sys.path.insert(0, '..')
from src.features import load_raw, build_features, get_X_y
from src.model import FailureClassifier, find_cost_optimal_threshold, business_cost, FN_COST, FP_COST

%matplotlib inline
sns.set_theme(style='darkgrid')

In [ ]:
df = build_features(load_raw())
X, y = get_X_y(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

clf = FailureClassifier.load('../models/')
proba = clf.predict_proba(X_test)
print(f"Optimal threshold from training: {clf.optimal_threshold}")

In [ ]:
# Business cost curve
thresholds = np.arange(0.05, 0.95, 0.025)
costs = []
for t in thresholds:
    preds = (proba >= t).astype(int)
    fn = int(((preds == 0) & (y_test.values == 1)).sum())
    fp = int(((preds == 1) & (y_test.values == 0)).sum())
    costs.append({'threshold': t, 'total_cost': fn*FN_COST + fp*FP_COST,
                  'fn_cost': fn*FN_COST, 'fp_cost': fp*FP_COST, 'fn': fn, 'fp': fp})

cost_df = pd.DataFrame(costs)
opt_t, opt_cost = find_cost_optimal_threshold(y_test.values, proba, thresholds)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(cost_df.threshold, cost_df.total_cost/1000, label='Total cost', linewidth=2)
ax.plot(cost_df.threshold, cost_df.fn_cost/1000, linestyle='--', label=f'FN cost (${FN_COST/1000:.0f}K each)')
ax.plot(cost_df.threshold, cost_df.fp_cost/1000, linestyle='--', label=f'FP cost (${FP_COST/1000:.0f}K each)')
ax.axvline(opt_t, color='gold', linestyle='-', alpha=0.8, label=f'Optimal: {opt_t:.2f}')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.6, label='Default: 0.50')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Total Cost ($K)')
ax.set_title('Business Cost vs. Threshold — Optimal threshold shifts from 0.5 to ~0.25')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/cost_vs_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrices: default vs optimal
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, t, label in zip(axes, [0.5, opt_t], ['Default (0.5)', f'Optimal ({opt_t:.2f})']):
    preds = (proba >= t).astype(int)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=['Pred No Fail', 'Pred Fail'],
                yticklabels=['Actual No Fail', 'Actual Fail'])
    cost = business_cost(y_test.values, preds)
    ax.set_title(f'{label}\nTotal cost: ${cost:,.0f}')

plt.tight_layout()
plt.savefig('../figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PR curve
precision_arr, recall_arr, thresh_arr = precision_recall_curve(y_test, proba)
pr_auc = average_precision_score(y_test, proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall_arr, precision_arr, label=f'PR-AUC = {pr_auc:.3f}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend()
plt.savefig('../figures/pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary at both thresholds
for t_name, t in [('Default 0.5', 0.5), (f'Optimal {opt_t:.2f}', opt_t)]:
    r = clf.evaluate(X_test, y_test, threshold=t)
    print(f"\n=== {t_name} ===")
    print(f"  Recall:    {r['recall']:.3f}  (caught {1-r['n_fn']/(r['n_fn']+sum(y_test==1)):.0%} of real failures)")
    print(f"  Precision: {r['precision']:.3f}")
    print(f"  F1:        {r['f1']:.3f}")
    print(f"  FN: {r['n_fn']}  FP: {r['n_fp']}  Business cost: ${r['business_cost']:,.0f}")